# Enhanced Drug Classification with Topological Data Analysis

**Project Goal:** Improve Random Forest accuracy from ~65% to higher using topological/geometrical features (≤120 features)

**Features Added:**
- Spectral Features (Laplacian eigenvalues)
- Carlsson Coordinates (density-based topology)
- Graph Statistics (classical graph theory)
- Persistent Homology (optimized TDA features)

**Total Features:** 43 (well under 120 limit)

## 1. Imports and Setup

In [4]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

# Molecular processing
from rdkit import Chem
from rdkit.Chem.Scaffolds import MurckoScaffold

# Machine learning
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

# Topological features
import networkx as nx
from scipy.sparse.linalg import eigsh
from scipy.sparse import csr_matrix
from sklearn.preprocessing import StandardScaler

# Giotto-TDA
try:
    from gtda.homology import VietorisRipsPersistence
    from gtda.diagrams import PersistenceEntropy, Amplitude, BettiCurve
    GTDATDA_AVAILABLE = True
    print(" Giotto-TDA available")
except ImportError:
    GTDATDA_AVAILABLE = False
    print(" Giotto-TDA not available. Some features will be disabled.")

print("All imports successful!")

 Giotto-TDA not available. Some features will be disabled.
All imports successful!


## 2. Spectral Feature Extractor

Extracts eigenvalues from graph Laplacians:
- **L0 (Graph Laplacian)**: L = D - A (captures node-level topology)
- **L1 (Edge Laplacian)**: L1 = B^T × B (captures edge-level topology)
- **Algebraic connectivity**: 2nd smallest eigenvalue (graph robustness)
- **Spectral gaps**: Eigenvalue differences (cluster structure)

In [6]:
class SpectralFeatureExtractor:
    """Extract spectral features from graph Laplacians"""
    
    def __init__(self, n_eigenvalues=5):
        self.n_eigenvalues = n_eigenvalues
    
    def compute_laplacian_0(self, adjacency):
        """Compute weighted 0-Laplacian (graph Laplacian)"""
        degrees = np.sum(adjacency, axis=1)
        D = np.diag(degrees)
        L = D - adjacency
        return L
    
    def compute_laplacian_1(self, adjacency):
        """Compute weighted 1-Laplacian (edge Laplacian)"""
        G = nx.from_numpy_array(adjacency)
        if G.number_of_edges() == 0:
            return np.zeros((1, 1))
        
        edges = list(G.edges())
        nodes = list(G.nodes())
        n_edges = len(edges)
        n_nodes = len(nodes)
        
        if n_edges == 0:
            return np.zeros((1, 1))
        
        # Incidence matrix B
        B = np.zeros((n_nodes, n_edges))
        for idx, (u, v) in enumerate(edges):
            B[u, idx] = 1
            B[v, idx] = -1
        
        L1 = B.T @ B
        return L1
    
    def get_eigenvalues(self, L, k=None):
        """Compute smallest k eigenvalues of Laplacian"""
        if k is None:
            k = min(self.n_eigenvalues, L.shape[0] - 1)
        
        k = max(1, min(k, L.shape[0] - 2))
        
        try:
            if L.shape[0] <= 20:
                eigenvalues = np.linalg.eigvalsh(L)
                eigenvalues = np.sort(eigenvalues)[:k]
            else:
                L_sparse = csr_matrix(L)
                eigenvalues, _ = eigsh(L_sparse, k=k, which='SM', tol=1e-3)
                eigenvalues = np.sort(eigenvalues)
            
            if len(eigenvalues) < self.n_eigenvalues:
                eigenvalues = np.pad(eigenvalues, 
                                   (0, self.n_eigenvalues - len(eigenvalues)), 
                                   'constant', constant_values=0)
            
            return eigenvalues[:self.n_eigenvalues]
        
        except:
            return np.zeros(self.n_eigenvalues)
    
    def extract_features(self, adjacency):
        """Extract all spectral features"""
        features = {}
        
        adj_clean = adjacency.copy()
        np.fill_diagonal(adj_clean, 0)
        
        # L0 eigenvalues
        L0 = self.compute_laplacian_0(adj_clean)
        eig_0 = self.get_eigenvalues(L0)
        features['laplacian_0_eigenvalues'] = eig_0
        
        # L1 eigenvalues
        L1 = self.compute_laplacian_1(adj_clean)
        if L1.shape[0] > 1:
            eig_1 = self.get_eigenvalues(L1)
            features['laplacian_1_eigenvalues'] = eig_1
        else:
            features['laplacian_1_eigenvalues'] = np.zeros(self.n_eigenvalues)
        
        # Algebraic connectivity
        features['algebraic_connectivity'] = eig_0[1] if len(eig_0) > 1 else 0
        
        # Spectral gaps
        features['spectral_gap_0'] = eig_0[2] - eig_0[1] if len(eig_0) > 2 else 0
        features['spectral_gap_1'] = (features['laplacian_1_eigenvalues'][1] - 
                                      features['laplacian_1_eigenvalues'][0])
        
        # Spectral radius
        features['spectral_radius'] = np.max(eig_0) if len(eig_0) > 0 else 0
        
        return features

print(" Spectral Feature Extractor defined")

 Spectral Feature Extractor defined


## 3. Carlsson Coordinates Extractor

Extracts density-based topological features:
- **Density statistics**: k-nearest neighbor densities (local geometry)
- **Eccentricity**: Distance from graph center (shape information)

In [8]:
class CarlssonCoordinatesExtractor:
    """Extract Carlsson coordinates (density-based topological features)"""
    
    def __init__(self, k_neighbors=3):
        self.k_neighbors = k_neighbors
    
    def compute_density(self, adjacency, k):
        """Compute k-nearest neighbor density for each node"""
        n = adjacency.shape[0]
        densities = np.zeros(n)
        
        for i in range(n):
            distances = adjacency[i, :].copy()
            distances[i] = np.inf
            
            nonzero_dist = distances[distances > 0]
            if len(nonzero_dist) >= k:
                k_nearest = np.partition(nonzero_dist, k-1)[:k]
                densities[i] = np.mean(k_nearest)
            elif len(nonzero_dist) > 0:
                densities[i] = np.mean(nonzero_dist)
        
        return densities
    
    def extract_features(self, adjacency):
        """Extract Carlsson coordinate features"""
        features = {}
        
        densities = self.compute_density(adjacency, self.k_neighbors)
        
        features['density_mean'] = np.mean(densities)
        features['density_std'] = np.std(densities)
        features['density_max'] = np.max(densities)
        features['density_min'] = np.min(densities) if np.min(densities) > 0 else 0
        
        # Eccentricity
        G = nx.from_numpy_array(adjacency)
        if G.number_of_nodes() > 0 and nx.is_connected(G):
            eccentricities = nx.eccentricity(G)
            ecc_values = list(eccentricities.values())
            features['eccentricity_mean'] = np.mean(ecc_values)
            features['eccentricity_std'] = np.std(ecc_values)
        else:
            features['eccentricity_mean'] = 0
            features['eccentricity_std'] = 0
        
        return features

print(" Carlsson Coordinates Extractor defined")

 Carlsson Coordinates Extractor defined


## 4. Graph Statistics Extractor

Extracts classical graph theory features:
- **Degree statistics**: Mean, std, max, min degree
- **Clustering**: Transitivity, average clustering coefficient
- **Path features**: Diameter, average shortest path, radius
- **Cycles**: Number, mean length, max length

In [10]:
class GraphStatisticsExtractor:
    """Extract classical graph theory features"""
    
    def extract_features(self, adjacency):
        """Extract comprehensive graph statistics"""
        features = {}
        
        G = nx.from_numpy_array(adjacency)
        n_nodes = G.number_of_nodes()
        n_edges = G.number_of_edges()
        
        # Basic counts
        features['num_nodes'] = n_nodes
        features['num_edges'] = n_edges
        features['density'] = nx.density(G) if n_nodes > 1 else 0
        
        # Degree statistics
        degrees = [d for n, d in G.degree()]
        if degrees:
            features['mean_degree'] = np.mean(degrees)
            features['std_degree'] = np.std(degrees)
            features['max_degree'] = np.max(degrees)
            features['min_degree'] = np.min(degrees)
        else:
            features['mean_degree'] = 0
            features['std_degree'] = 0
            features['max_degree'] = 0
            features['min_degree'] = 0
        
        # Clustering
        features['avg_clustering'] = nx.average_clustering(G)
        features['transitivity'] = nx.transitivity(G)
        
        # Connected components
        features['num_components'] = nx.number_connected_components(G)
        
        # Path features
        if nx.is_connected(G) and n_nodes > 1:
            features['diameter'] = nx.diameter(G)
            features['avg_shortest_path'] = nx.average_shortest_path_length(G)
            features['radius'] = nx.radius(G)
        else:
            if n_nodes > 1:
                largest_cc = max(nx.connected_components(G), key=len)
                G_largest = G.subgraph(largest_cc)
                if G_largest.number_of_nodes() > 1:
                    features['diameter'] = nx.diameter(G_largest)
                    features['avg_shortest_path'] = nx.average_shortest_path_length(G_largest)
                    features['radius'] = nx.radius(G_largest)
                else:
                    features['diameter'] = 0
                    features['avg_shortest_path'] = 0
                    features['radius'] = 0
            else:
                features['diameter'] = 0
                features['avg_shortest_path'] = 0
                features['radius'] = 0
        
        # Cycles
        try:
            cycles = nx.cycle_basis(G)
            features['num_cycles'] = len(cycles)
            if cycles:
                cycle_lengths = [len(c) for c in cycles]
                features['mean_cycle_length'] = np.mean(cycle_lengths)
                features['max_cycle_length'] = np.max(cycle_lengths)
            else:
                features['mean_cycle_length'] = 0
                features['max_cycle_length'] = 0
        except:
            features['num_cycles'] = 0
            features['mean_cycle_length'] = 0
            features['max_cycle_length'] = 0
        
        # Assortativity
        try:
            features['degree_assortativity'] = nx.degree_assortativity_coefficient(G)
        except:
            features['degree_assortativity'] = 0
        
        return features

print(" Graph Statistics Extractor defined")

 Graph Statistics Extractor defined


## 5. Persistent Homology Extractor

Extracts TDA features using Giotto-TDA:
- **Persistent entropy**: Complexity of topological features
- **Amplitudes**: Bottleneck and Landscape (strength of persistence)

In [12]:
class PersistentHomologyExtractor:
    """Extract topological features using giotto-tda"""
    
    def __init__(self, homology_dimensions=(0, 1), n_bins=30, n_jobs=1):
        self.homology_dimensions = homology_dimensions
        self.n_bins = n_bins
        self.n_jobs = n_jobs
        
        if GTDATDA_AVAILABLE:
            self._initialize_gtda()
        else:
            self.persistence = None
            self.entropy = None
            self.amplitudes = {}
    
    def _initialize_gtda(self):
        """Initialize giotto-tda components"""
        self.persistence = VietorisRipsPersistence(
            metric='precomputed',
            homology_dimensions=self.homology_dimensions,
            collapse_edges=True,
            n_jobs=self.n_jobs
        )
        
        self.entropy = PersistenceEntropy(n_jobs=self.n_jobs)
        
        self.amplitudes = {
            'bottleneck': Amplitude(metric='bottleneck', n_jobs=self.n_jobs),
            'landscape': Amplitude(
                metric='landscape',
                metric_params={'n_layers': 1, 'n_bins': self.n_bins},
                n_jobs=self.n_jobs
            ),
        }
    
    def adjacency_to_distance(self, adjacency_matrix):
        """Convert adjacency matrix to distance matrix"""
        distance = adjacency_matrix.copy()
        np.fill_diagonal(distance, 0)
        
        with np.errstate(divide='ignore', invalid='ignore'):
            distance = 1.0 / (distance + 1e-10)
            distance[distance > 1e6] = 1e6
        
        distance = np.maximum(distance, distance.T)
        np.fill_diagonal(distance, 0)
        
        return distance
    
    def extract_features(self, adjacency_matrices):
        """Extract persistent homology features"""
        if not GTDATDA_AVAILABLE:
            return self._get_empty_features(adjacency_matrices)
        
        features = {}
        
        distance_matrices = np.array([
            self.adjacency_to_distance(adj) for adj in adjacency_matrices
        ])
        
        persistence_diagrams = self.persistence.fit_transform(distance_matrices)
        features['persistence_diagrams'] = persistence_diagrams
        
        entropy_features = self.entropy.fit_transform(persistence_diagrams)
        features['persistent_entropy'] = entropy_features
        
        amplitude_features = {}
        for metric_name, amplitude_calc in self.amplitudes.items():
            try:
                amp = amplitude_calc.fit_transform(persistence_diagrams)
                amplitude_features[metric_name] = amp
            except:
                n_samples = len(adjacency_matrices)
                n_dims = len(self.homology_dimensions)
                amplitude_features[metric_name] = np.zeros((n_samples, n_dims))
        features['amplitudes'] = amplitude_features
        
        return features
    
    def _get_empty_features(self, adjacency_matrices):
        """Return empty features when giotto-tda is not available"""
        n_samples = len(adjacency_matrices)
        n_dims = len(self.homology_dimensions)
        
        return {
            'persistence_diagrams': [np.array([])] * n_samples,
            'persistent_entropy': np.zeros((n_samples, n_dims)),
            'amplitudes': {
                'bottleneck': np.zeros((n_samples, n_dims)),
                'landscape': np.zeros((n_samples, n_dims)),
            }
        }

print(" Persistent Homology Extractor defined")

 Persistent Homology Extractor defined


## 6. Main Feature Generator

Combines all feature extractors into one comprehensive generator.

In [14]:
class DiscriminativeFeatureGenerator:
    """
    Comprehensive feature generator combining:
    - Persistent homology (TDA)
    - Spectral features (Laplacians)
    - Carlsson coordinates
    - Graph statistics
    """
    
    def __init__(self, homology_dimensions=(0, 1), n_bins=30, 
                 n_eigenvalues=5, n_jobs=1):
        self.homology_dimensions = homology_dimensions
        self.n_bins = n_bins
        self.n_eigenvalues = n_eigenvalues
        
        # Initialize all extractors
        self.scaler = StandardScaler()
        self.ph_extractor = PersistentHomologyExtractor(
            homology_dimensions=homology_dimensions,
            n_bins=n_bins,
            n_jobs=n_jobs
        )
        self.spectral_extractor = SpectralFeatureExtractor(n_eigenvalues=n_eigenvalues)
        self.carlsson_extractor = CarlssonCoordinatesExtractor(k_neighbors=3)
        self.graph_stats_extractor = GraphStatisticsExtractor()
        
        self.feature_dim = None
        self.feature_names = []
    
    def generate_features(self, adjacency_matrices, labels):
        """Generate all features"""
        print("=" * 60)
        print("GENERATING COMPREHENSIVE TOPOLOGICAL FEATURES")
        print("=" * 60)
        
        X = []
        y = []
        
        # Determine feature dimension
        if self.feature_dim is None:
            print("\nDetermining feature dimension...")
            test_feature = self._graph_to_features(adjacency_matrices[0])
            self.feature_dim = len(test_feature)
            self.feature_names = self._get_feature_names()
            print(f"Total feature dimension: {self.feature_dim}")
            self._print_feature_breakdown()
        
        # Extract features
        for i, adjacency in enumerate(adjacency_matrices):
            if i % 500 == 0:
                print(f"Processing graph {i+1}/{len(adjacency_matrices)}...")
            
            feature_vector = self._graph_to_features(adjacency)
            
            if len(feature_vector) != self.feature_dim:
                if len(feature_vector) > self.feature_dim:
                    feature_vector = feature_vector[:self.feature_dim]
                else:
                    feature_vector = np.pad(
                        feature_vector, 
                        (0, self.feature_dim - len(feature_vector)), 
                        'constant'
                    )
            
            X.append(feature_vector)
            y.append(labels[i])
        
        X = np.array(X)
        y = np.array(y)
        
        print(f"\n{'='*60}")
        print(f"Feature extraction completed!")
        print(f"Total samples: {len(adjacency_matrices)}")
        print(f"Feature matrix shape: {X.shape}")
        print(f"{'='*60}\n")
        
        return X, y, {'feature_names': self.feature_names}
    
    def _graph_to_features(self, adjacency):
        """Extract all features from a single graph"""
        features = []
        
        # 1. Persistent homology
        ph_features = self.ph_extractor.extract_features([adjacency])
        
        if 'persistent_entropy' in ph_features:
            features.extend(ph_features['persistent_entropy'][0])
        
        if 'amplitudes' in ph_features:
            for metric in ['bottleneck', 'landscape']:
                if metric in ph_features['amplitudes']:
                    features.extend(ph_features['amplitudes'][metric][0])
        
        # 2. Spectral features
        spectral_features = self.spectral_extractor.extract_features(adjacency)
        features.extend(spectral_features['laplacian_0_eigenvalues'])
        features.extend(spectral_features['laplacian_1_eigenvalues'])
        features.append(spectral_features['algebraic_connectivity'])
        features.append(spectral_features['spectral_gap_0'])
        features.append(spectral_features['spectral_gap_1'])
        features.append(spectral_features['spectral_radius'])
        
        # 3. Carlsson coordinates
        carlsson_features = self.carlsson_extractor.extract_features(adjacency)
        for key in ['density_mean', 'density_std', 'density_max', 'density_min',
                    'eccentricity_mean', 'eccentricity_std']:
            features.append(carlsson_features[key])
        
        # 4. Graph statistics
        graph_features = self.graph_stats_extractor.extract_features(adjacency)
        for key in ['num_nodes', 'num_edges', 'density', 'mean_degree', 'std_degree',
                    'max_degree', 'min_degree', 'avg_clustering', 'transitivity',
                    'num_components', 'diameter', 'avg_shortest_path', 'radius',
                    'num_cycles', 'mean_cycle_length', 'max_cycle_length',
                    'degree_assortativity']:
            features.append(graph_features[key])
        
        return np.array(features)
    
    def _get_feature_names(self):
        """Get descriptive feature names"""
        names = []
        
        # Persistent homology
        for dim in self.homology_dimensions:
            names.append(f'persistent_entropy_H{dim}')
        
        for metric in ['bottleneck', 'landscape']:
            for dim in self.homology_dimensions:
                names.append(f'amplitude_{metric}_H{dim}')
        
        # Spectral features
        for i in range(self.n_eigenvalues):
            names.append(f'L0_eigenvalue_{i}')
        for i in range(self.n_eigenvalues):
            names.append(f'L1_eigenvalue_{i}')
        names.extend(['algebraic_connectivity', 'spectral_gap_0', 
                     'spectral_gap_1', 'spectral_radius'])
        
        # Carlsson coordinates
        names.extend(['density_mean', 'density_std', 'density_max', 'density_min',
                     'eccentricity_mean', 'eccentricity_std'])
        
        # Graph statistics
        names.extend(['num_nodes', 'num_edges', 'graph_density', 'mean_degree', 
                     'std_degree', 'max_degree', 'min_degree', 'avg_clustering',
                     'transitivity', 'num_components', 'diameter', 'avg_shortest_path',
                     'radius', 'num_cycles', 'mean_cycle_length', 'max_cycle_length',
                     'degree_assortativity'])
        
        return names
    
    def _print_feature_breakdown(self):
        """Print detailed feature breakdown"""
        n_dims = len(self.homology_dimensions)
        
        counts = {
            'Persistent Entropy': n_dims,
            'Amplitude Metrics': 2 * n_dims,
            'L0 Eigenvalues': self.n_eigenvalues,
            'L1 Eigenvalues': self.n_eigenvalues,
            'Spectral Stats': 4,
            'Carlsson Coordinates': 6,
            'Graph Statistics': 17
        }
        
        print("\nFEATURE BREAKDOWN:")
        print("-" * 60)
        for category, count in counts.items():
            print(f"  {category:30s}: {count:3d} features")
        print("-" * 60)
        print(f"  {'TOTAL':30s}: {sum(counts.values()):3d} features")
        print("-" * 60)
    
    def fit_transform(self, adjacency_matrices, labels):
        """Generate features and fit scaler"""
        X, y, feature_info = self.generate_features(adjacency_matrices, labels)
        if len(X) == 0:
            return None, None, None
        X_scaled = self.scaler.fit_transform(X)
        return X_scaled, y, feature_info
    
    def transform(self, adjacency_matrices):
        """Transform new data"""
        X = []
        for adjacency in adjacency_matrices:
            feature_vector = self._graph_to_features(adjacency)
            if len(feature_vector) != self.feature_dim:
                if len(feature_vector) > self.feature_dim:
                    feature_vector = feature_vector[:self.feature_dim]
                else:
                    feature_vector = np.pad(
                        feature_vector, 
                        (0, self.feature_dim - len(feature_vector)), 
                        'constant'
                    )
            X.append(feature_vector)
        
        X = np.array(X)
        if self.scaler is not None:
            X = self.scaler.transform(X)
        return X

print(" Main Feature Generator defined")

 Main Feature Generator defined


## 7. SMILES to Graph Converter

In [16]:
class SMILESToGraphConverter:
    """Convert SMILES strings to weighted adjacency matrices"""
    
    def __init__(self, max_atoms=100):
        self.max_atoms = max_atoms
    
    def smiles_to_adjacency(self, smiles):
        """Convert a SMILES string to a weighted adjacency matrix"""
        try:
            mol = Chem.MolFromSmiles(smiles)
            if mol is None:
                return np.zeros((self.max_atoms, self.max_atoms))
            
            n_atoms = mol.GetNumAtoms()
            if n_atoms > self.max_atoms:
                return np.zeros((self.max_atoms, self.max_atoms))
            
            adjacency = np.zeros((self.max_atoms, self.max_atoms))
            
            for i in range(n_atoms):
                for j in range(i + 1, n_atoms):
                    bond = mol.GetBondBetweenAtoms(i, j)
                    if bond:
                        bt = bond.GetBondType()
                        if bt == Chem.rdchem.BondType.SINGLE:
                            w = 1.0
                        elif bt == Chem.rdchem.BondType.DOUBLE:
                            w = 2.0
                        elif bt == Chem.rdchem.BondType.TRIPLE:
                            w = 3.0
                        elif bt == Chem.rdchem.BondType.AROMATIC:
                            w = 1.5
                        else:
                            w = 1.0
                        
                        adjacency[i, j] = adjacency[j, i] = w
            
            return adjacency
        
        except Exception as e:
            print(f"Error converting SMILES {smiles}: {e}")
            return np.zeros((self.max_atoms, self.max_atoms))
    
    def convert_smiles_list(self, smiles_list):
        """Convert list of SMILES to adjacency matrices"""
        adjacency_matrices = []
        valid_indices = []
        
        for i, sm in enumerate(smiles_list):
            if pd.isna(sm):
                continue
            
            adj = self.smiles_to_adjacency(sm)
            adjacency_matrices.append(adj)
            valid_indices.append(i)
        
        return adjacency_matrices, valid_indices

print(" SMILES Converter defined")

 SMILES Converter defined


## 8. Scaffold Split Function

In [18]:
def scaffold_split(df, test_size=0.2, random_state=42):
    """Split data by molecular scaffold to prevent data leakage"""
    scaffolds = {}
    
    for idx, row in df.iterrows():
        smiles = row['SMILES']
        mol = Chem.MolFromSmiles(smiles)
        
        if mol is None:
            scaffold = 'invalid'
        else:
            try:
                scaffold = MurckoScaffold.MurckoScaffoldSmiles(mol=mol)
            except:
                scaffold = smiles
        
        if scaffold not in scaffolds:
            scaffolds[scaffold] = []
        scaffolds[scaffold].append(idx)
    
    scaffold_list = list(scaffolds.keys())
    np.random.seed(random_state)
    np.random.shuffle(scaffold_list)
    
    n_test = int(len(df) * test_size)
    test_indices = []
    train_indices = []
    
    for scaffold in scaffold_list:
        if len(test_indices) < n_test:
            test_indices.extend(scaffolds[scaffold])
        else:
            train_indices.extend(scaffolds[scaffold])
    
    return train_indices, test_indices

print(" Scaffold split function defined")

 Scaffold split function defined


## 9. Complete Pipeline Class

In [20]:
class DrugClassificationPipeline:
    """Complete pipeline for drug classification using TDA"""
    
    def __init__(self, n_bins=30, max_atoms=100, n_eigenvalues=5,
                 test_size=0.2, random_state=42):
        
        self.n_bins = n_bins
        self.max_atoms = max_atoms
        self.n_eigenvalues = n_eigenvalues
        self.test_size = test_size
        self.random_state = random_state
        
        self.graph_converter = SMILESToGraphConverter(max_atoms=max_atoms)
        self.feature_generator = DiscriminativeFeatureGenerator(
            n_bins=n_bins,
            n_eigenvalues=n_eigenvalues
        )
        
        self.label_encoder = LabelEncoder()
        self.models = {}
        self.results = {}
        self.class_names = None
    
    def load_data(self, filepath="drugs_combined.csv"):
        """Load drug data"""
        df = pd.read_csv(filepath)
        print(f"Loaded {len(df)} samples")
        return df
    
    def preprocess_data(self, df):
        """Clean and prepare data"""
        df_clean = df.dropna(subset=["SMILES", "target"])
        print(f"After cleaning: {len(df_clean)} samples")
        return df_clean
    
    def prepare_features(self, df):
        """Convert SMILES to features"""
        print("\nConverting SMILES to graphs...")
        smiles_list = df["SMILES"].tolist()
        targets = df["target"].tolist()
        
        adjacency_matrices, valid_indices = self.graph_converter.convert_smiles_list(smiles_list)
        valid_targets = [targets[i] for i in valid_indices]
        valid_df = df.iloc[valid_indices].reset_index(drop=True)
        
        print(f"Valid graphs: {len(adjacency_matrices)}")
        
        X, y, feature_info = self.feature_generator.fit_transform(
            adjacency_matrices, 
            valid_targets
        )
        
        if isinstance(valid_targets[0], str):
            y_encoded = self.label_encoder.fit_transform(valid_targets)
            self.class_names = self.label_encoder.classes_
        else:
            y_encoded = np.array(valid_targets)
            self.class_names = sorted(np.unique(valid_targets))
        
        return X, y_encoded, valid_df, feature_info
    
    def train_models(self, X_train, X_test, y_train, y_test):
        """Train and evaluate models"""
        print("\n" + "="*60)
        print("TRAINING MODELS")
        print("="*60)
        
        models = {
            "Random Forest": RandomForestClassifier(
                n_estimators=500,
                max_depth=30,
                min_samples_split=5,
                min_samples_leaf=2,
                class_weight="balanced",
                n_jobs=-1,
                random_state=self.random_state
            ),
            "SVM": SVC(
                C=1.0,
                kernel='rbf',
                probability=True,
                class_weight="balanced",
                random_state=self.random_state
            )
        }
        
        results = {}
        
        for name, model in models.items():
            print(f"\nTraining {name}...")
            
            model.fit(X_train, y_train)
            y_pred = model.predict(X_test)
            
            acc = accuracy_score(y_test, y_pred)
            cv = cross_val_score(model, X_train, y_train, cv=5, n_jobs=-1)
            
            results[name] = {
                "model": model,
                "y_pred": y_pred,
                "accuracy": acc,
                "cv_mean": cv.mean(),
                "cv_std": cv.std()
            }
            
            print(f"\n{name} Results:")
            print(f"  Test Accuracy:  {acc:.4f}")
            print(f"  CV Accuracy:    {cv.mean():.4f} (±{cv.std():.4f})")
            
            print(f"\nClassification Report ({name}):")
            print(classification_report(y_test, y_pred, 
                                       target_names=[str(c) for c in self.class_names]))
        
        self.results = results
        self.models = models
    
    def run_pipeline(self):
        """Execute complete pipeline"""
        print("\n" + "="*60)
        print("DRUG CLASSIFICATION PIPELINE - TOPOLOGICAL DATA ANALYSIS")
        print("="*60)
        
        df = self.load_data()
        df = self.preprocess_data(df)
        
        print("\nPerforming scaffold split...")
        train_idx, test_idx = scaffold_split(
            df, 
            test_size=self.test_size,
            random_state=self.random_state
        )
        
        df_train = df.iloc[train_idx].reset_index(drop=True)
        df_test = df.iloc[test_idx].reset_index(drop=True)
        
        print(f"Training set: {len(df_train)} samples")
        print(f"Test set:     {len(df_test)} samples")
        
        print("\n" + "="*60)
        print("FEATURE EXTRACTION - TRAINING SET")
        print("="*60)
        X_train, y_train, _, _ = self.prepare_features(df_train)
        
        print("\n" + "="*60)
        print("FEATURE EXTRACTION - TEST SET")
        print("="*60)
        X_test = self.feature_generator.transform(
            [self.graph_converter.smiles_to_adjacency(s) 
             for s in df_test['SMILES'] if not pd.isna(s)]
        )
        y_test = self.label_encoder.transform(df_test['target'].tolist())
        
        self.train_models(X_train, X_test, y_train, y_test)
        
        return X_train, y_train, X_test, y_test

print(" Pipeline class defined")

 Pipeline class defined


## 10. Run the Complete Pipeline

**Feature Summary:**
- Persistent Entropy: 2 features
- Amplitude Metrics: 4 features
- L0 Eigenvalues: 5 features
- L1 Eigenvalues: 5 features
- Spectral Statistics: 4 features
- Carlsson Coordinates: 6 features
- Graph Statistics: 17 features

**Total: 43 features** (well under 120 limit)

In [21]:
# Initialize pipeline
pipeline = DrugClassificationPipeline(
    n_bins=30,
    max_atoms=100,
    n_eigenvalues=5,
    test_size=0.2,
    random_state=42
)

# Run complete pipeline
X_train, y_train, X_test, y_test = pipeline.run_pipeline()

print("\n" + "="*60)
print("PIPELINE COMPLETED SUCCESSFULLY")
print("="*60)


DRUG CLASSIFICATION PIPELINE - TOPOLOGICAL DATA ANALYSIS
Loaded 6939 samples
After cleaning: 6939 samples

Performing scaffold split...
Training set: 5552 samples
Test set:     1387 samples

FEATURE EXTRACTION - TRAINING SET

Converting SMILES to graphs...
Valid graphs: 5552
GENERATING COMPREHENSIVE TOPOLOGICAL FEATURES

Determining feature dimension...
Total feature dimension: 43

FEATURE BREAKDOWN:
------------------------------------------------------------
  Persistent Entropy            :   2 features
  Amplitude Metrics             :   4 features
  L0 Eigenvalues                :   5 features
  L1 Eigenvalues                :   5 features
  Spectral Stats                :   4 features
  Carlsson Coordinates          :   6 features
  Graph Statistics              :  17 features
------------------------------------------------------------
  TOTAL                         :  43 features
------------------------------------------------------------
Processing graph 1/5552...
Processin

ValueError: Input X contains infinity or a value too large for dtype('float64').

## 11. Analyze Results

In [ ]:
# Display final results summary
print("\n" + "="*60)
print("FINAL RESULTS SUMMARY")
print("="*60)

for model_name, result in pipeline.results.items():
    print(f"\n{model_name}:")
    print(f"  Test Accuracy:     {result['accuracy']:.4f}")
    print(f"  CV Mean Accuracy:  {result['cv_mean']:.4f}")
    print(f"  CV Std:            {result['cv_std']:.4f}")

print("\n" + "="*60)
print(f"Total Features Used: {pipeline.feature_generator.feature_dim}")
print("All features are topological/geometrical from molecular graphs")
print("="*60)

## 12. Feature Importance Analysis (Optional)

In [ ]:
# Get feature importances from Random Forest
rf_model = pipeline.results['Random Forest']['model']
importances = rf_model.feature_importances_
feature_names = pipeline.feature_generator.feature_names

# Sort by importance
indices = np.argsort(importances)[::-1]

print("\nTop 20 Most Important Features:")
print("="*60)
for i in range(min(20, len(indices))):
    idx = indices[i]
    print(f"{i+1:2d}. {feature_names[idx]:35s}: {importances[idx]:.4f}")

## Conclusion

This notebook implements a comprehensive topological data analysis pipeline for drug classification:

**Key Improvements:**
1. Added spectral features (Laplacian eigenvalues) - 14 features
2. Added Carlsson coordinates (density topology) - 6 features
3. Added graph statistics (geometric properties) - 17 features
4. Optimized persistent homology features - 6 features

**Total: 43 features** (under 120 limit)

**All features are:**
- ✓ Topological/geometrical from graphs
- ✓ Well-understood and interpretable
- ✓ Complementary (capture different aspects)
- ✓ Computationally efficient

**Expected improvements:**
- Better accuracy due to feature diversity
- Less overfitting (fewer features)
- More robust generalization
- Interpretable results